In [10]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score


In [11]:
Z_train = np.load("../artifacts/data/Z_train.npy")
Z_test  = np.load("../artifacts/data/Z_test.npy")

y_train = np.load("../artifacts/data/y_train.npy")
y_test  = np.load("../artifacts/data/y_test.npy")


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Z_train_t = torch.tensor(Z_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

Z_test_t  = torch.tensor(Z_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.float32)


In [13]:
class EmbeddingClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze()


In [14]:
input_dim = Z_train.shape[1]

model = EmbeddingClassifier(input_dim).to(device)

model.load_state_dict(
    torch.load("../artifacts/models/classifier_embeddings.pth",
               map_location=device)
)

model.train()


EmbeddingClassifier(
  (net): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [15]:
#utilizo o ReplayBuffer para manter uma amostra dos ataques passados
#para que não haja esquecimento catastrófico ao treinar o modelo com dados novos

class ReplayBuffer:

    def __init__(self, max_size=5000): #equilibrado
        self.max_size = max_size
        self.X = []
        self.y = []

    def add(self, X, y):
        for i in range(len(X)):
            if len(self.X) >= self.max_size:
                self.X.pop(0) #lógica FIFO, mantenho uma janela histórica sempre que o modelo se adapta ao novo
                self.y.pop(0)

            self.X.append(X[i])
            self.y.append(y[i])

    def sample(self, batch_size):
        idx = np.random.choice(len(self.X), batch_size, replace=False)
        return (
            torch.stack([self.X[i] for i in idx]), #transformo a lista de tensores individuais em um único bath antes de enviar para gpu
            torch.stack([self.y[i] for i in idx])
        )


In [16]:
#otimizo a VRAM dividindo os embeddings em chunks

chunk_size = 20000

chunks_X = torch.split(Z_train_t, chunk_size)
chunks_y = torch.split(y_train_t, chunk_size)


In [17]:
#modelo aprende com o agora (chunk) mas constantemente lembra do que aconteceu no passado (buffer)
#na proporção 20k:1k
#resumo: o modelo foi treinado com o dataset UNSW, mas consegue se adaptar à medida que novos pacotes chegam
#é capaz de se ajustar com as particularidades de cada rede
#replaybuffer garante que detecta os novos sem esquecer os clássicos do dataset
#resultados: graças o replaybuffer o modelo consegue se adaptar

buffer = ReplayBuffer(max_size=5000)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

for t, (X_chunk, y_chunk) in enumerate(zip(chunks_X, chunks_y)):

    print(f"\n=== Step {t+1} ===")

    X_chunk = X_chunk.to(device)
    y_chunk = y_chunk.to(device)

    buffer.add(X_chunk.cpu(), y_chunk.cpu()) # adiciona ao buffer

    # replay
    if len(buffer.X) > 1024:
        X_old, y_old = buffer.sample(1024)
        X_old = X_old.to(device)
        y_old = y_old.to(device)

        X_train_step = torch.cat([X_chunk, X_old])
        y_train_step = torch.cat([y_chunk, y_old])

    else:
        X_train_step = X_chunk
        y_train_step = y_chunk

    logits = model(X_train_step).squeeze()
    loss = criterion(logits, y_train_step)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("Loss:", loss.item())



=== Step 1 ===
Loss: 0.018621662631630898

=== Step 2 ===
Loss: 0.00815759226679802

=== Step 3 ===
Loss: 0.2952521741390228

=== Step 4 ===
Loss: 0.20557786524295807

=== Step 5 ===
Loss: 0.19778472185134888

=== Step 6 ===
Loss: 0.142912358045578

=== Step 7 ===
Loss: 0.06658782064914703

=== Step 8 ===
Loss: 0.04288087412714958

=== Step 9 ===
Loss: 0.04337751865386963


In [18]:
#resultado: modelo não se degradou

model.eval()

with torch.no_grad():
    logits = model(Z_test_t.to(device)).squeeze()
    probs = torch.sigmoid(logits).cpu().numpy()

auc = roc_auc_score(y_test, probs)

print(f"ROC AUC pós-continual: {auc:.3}")


ROC AUC pós-continual: 0.974
